In [20]:
from __future__ import annotations

import argparse
import shutil
from pathlib import Path

import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
print("Jax version", jax.__version__, "64-bit", jax.config.read("jax_enable_x64"))
import s2fft
from tqdm.auto import tqdm


def resample_mw_to_mwss(map_path: Path, output_path: Path) -> None:
    m = np.load(map_path)
    m = m.reshape((1500, 2999, 5)).astype(np.float32)

    m_mwss = np.stack(
        [
            np.array(s2fft.utils.resampling.mw_to_mwss(m[..., c], L=1500)).real
            for c in range(5)
        ],
        axis=-1,
    )

    np.save(output_path, m_mwss.astype(np.float32, copy=False))

def L_downsample_by_two_mwss(map_path: Path, output_path: Path, L_in: int) -> None:
    m = np.load(map_path)
    m_halfL = np.stack(
            [
                np.array(s2fft.utils.resampling.downsample_by_two_mwss(m[..., c], L=L_in)).real
                for c in range(5)
            ], 
        axis=-1
        )  # shape: (L_in/2, ...)

    np.save(output_path, m_halfL.astype(np.float32, copy=False))



Jax version 0.11.0 64-bit True


In [21]:
SOURCE_DIR = Path(
    "/projects/u6pf/brianycc/spherical_maps/"
    "5_tomo_bins_flm_1500_batch_1_mwss/"
)
OUTPUT_DIR = Path(
    "/projects/u6pf/brianycc/spherical_maps/"
    "5_tomo_bins_flm_750_batch_1_mwss/"
)
OVERWRITE = False


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    params_src = SOURCE_DIR / "params.npy"
    params_dst = OUTPUT_DIR / "params.npy"
    if params_src.exists() and (OVERWRITE or not params_dst.exists()):
        shutil.copy2(params_src, params_dst)

    map_files = sorted(SOURCE_DIR.glob("map_*.npy"))
    if not map_files:
        raise FileNotFoundError(f"No map_*.npy files found in {SOURCE_DIR}")

    converted = 0
    skipped = 0

    for map_path in tqdm(map_files, desc="Resampling maps"):
        output_path = OUTPUT_DIR / map_path.name
        if output_path.exists() and not OVERWRITE:
            skipped += 1
            continue

        # resample_mw_to_mwss(map_path, output_path)
        L_downsample_by_two_mwss(map_path, output_path, L_in=1500)
        converted += 1

    print(
        f"Converted {converted} maps, skipped {skipped} existing maps.\n"
        f"Source: {SOURCE_DIR}\n"
        f"Output: {OUTPUT_DIR}"
    )

print("SOURCE_DIR =", SOURCE_DIR)
print("Exists     =", SOURCE_DIR.exists())
print("Is dir     =", SOURCE_DIR.is_dir())
map_files = sorted(SOURCE_DIR.glob("map_*.npy"))
print("Count      =", len(map_files))
print("First 5    =", [p.name for p in map_files[:5]])

main()

SOURCE_DIR = /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_1500_batch_1_mwss
Exists     = True
Is dir     = True
Count      = 2000
First 5    = ['map_0.npy', 'map_1.npy', 'map_10.npy', 'map_100.npy', 'map_1000.npy']


Resampling maps:   0%|                                                                                                               | 0/2000 [00:00<?, ?it/s]

Resampling maps: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:00<00:00, 43320.86it/s]

Converted 0 maps, skipped 2000 existing maps.
Source: /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_1500_batch_1_mwss
Output: /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_batch_1_mwss


In [14]:
check_path = OUTPUT_DIR / "map_0.npy"
m = np.load(check_path)
print("Check path:", check_path)
print("Shape:", m.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_batch_1_mwss/map_0.npy'

In [ ]:
# def main() -> None:
#     parser = argparse.ArgumentParser(
#         description="Bulk-resample spherical maps from MW to MWSS."
#     )
#     parser.add_argument(
#         "--source-dir",
#         type=Path,
#         default=Path(
#             "/projects/u6pf/brianycc/spherical_maps/"
#             "map_compression_samples_5_tomo_bins_flm_1500_batch_1"
#         ),
#         help="Directory containing map_*.npy and params.npy",
#     )
#     parser.add_argument(
#         "--output-dir",
#         type=Path,
#         default=Path(
#             "/projects/u6pf/brianycc/spherical_maps/"
#             "map_compression_samples_5_tomo_bins_flm_1500_batch_1_mwss"
#         ),
#         help="Directory where MWSS-resampled files will be written",
#     )
#     args = parser.parse_args()

#     source_dir: Path = args.source_dir
#     output_dir: Path = args.output_dir
#     output_dir.mkdir(parents=True, exist_ok=True)

#     params_src = source_dir / "params.npy"
#     params_dst = output_dir / "params.npy"
#     if params_src.exists() and not params_dst.exists():
#         shutil.copy2(params_src, params_dst)

#     map_files = sorted(source_dir.glob("map_*.npy"))
#     if not map_files:
#         raise FileNotFoundError(f"No map_*.npy files found in {source_dir}")

#     skipped = 0
#     converted = 0

#     for map_path in map_files:
#         output_path = output_dir / map_path.name
#         if output_path.exists():
#             skipped += 1
#             continue
#         resample_mw_to_mwss(map_path, output_path)
#         converted += 1

#     print(
#         f"Converted {converted} maps, skipped {skipped} existing maps "
#         f"from {source_dir} to {output_dir}"
#     )


# if __name__ == "__main__":
#     main()